# Batch Correction Visualization Pipeline

This notebook provides a step-by-step interactive visualization for evaluating **Batch Correction** results.
It compares **Time-Corrected + Arcsinh** data (Before Batch Correction) with **Batch-Corrected** data (After Batch Correction).

#### This is for the visualization of the Batch-corrected results

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os
import sys
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from collections import defaultdict
import collections.abc
collections.MutableMapping = collections.abc.MutableMapping
import scanpy as sc
import seaborn as sns
import anndata as ad
from matplotlib.lines import Line2D
%matplotlib inline
# Setup path to import local CyTof_BatchCorrection package
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils import *
from src.batch_correction_visualization import *
from src.time_correction_visualization import *
print(f"Project root: {project_root}")

In [ ]:
# ---- USER INPUT -----------------------------------------------------------
# Point to any compatible study config (project method, CytoNorm, CytofRUV, etc.).
study_config_path = "../results/study_config.json"
# Optional explicit selection. Use None to take dataset/panel from the study config.
target_dataset_override = None       # e.g. "D0", "D24", or None
target_panel_override   = None       # e.g. "Panel1", "Panel2", or None

# Chunked reading prevents loading a merged matrix before filtering.
matrix_chunksize = 100_000
# Decide from the original filtered row count whether to load all cells or sample.
matrix_sampling_threshold = 10_000_000
# Above the threshold, keep at most N cells per sample_name + plate.
panel_sample_per_group = 20000
panel_sample_group_cols = ("sample_name", "plate")
# --------------------------------------------------------------------------

def read_matrix_filtered_grouped(
    path,
    dataset=None,
    panel=None,
    chunksize=250_000,
    sample_per_group=None,
    group_cols=("sample_name", "plate"),
    random_state=42,
    low_memory=False,
):
    """
    Read a matrix CSV with optional dataset/panel filtering and balanced streaming sampling.

    Unlike read_matrix_filtered(..., sample_per_sample=...), this samples per group
    such as sample_name + plate, so repeated anchor/sample names are preserved across plates.
    """
    if sample_per_group is None or sample_per_group <= 0:
        return read_matrix_filtered(
            path,
            dataset=dataset,
            panel=panel,
            chunksize=chunksize,
            sample_per_sample=None,
            random_state=random_state,
            low_memory=low_memory,
        )

    if chunksize is None or chunksize <= 0:
        df = read_matrix_filtered(
            path,
            dataset=dataset,
            panel=panel,
            chunksize=None,
            sample_per_sample=None,
            random_state=random_state,
            low_memory=low_memory,
        )
        present_group_cols = [c for c in group_cols if c in df.columns]
        if not present_group_cols:
            return df.sample(n=min(len(df), sample_per_group), random_state=random_state).reset_index(drop=True)
        return (
            df.groupby(present_group_cols, observed=True, group_keys=False)
              .apply(lambda g: g.sample(n=min(len(g), sample_per_group), random_state=random_state))
              .reset_index(drop=True)
        )

    kept = []
    counts = {}
    rng_seed = int(random_state)

    for chunk_idx, chunk in enumerate(pd.read_csv(path, chunksize=chunksize, low_memory=low_memory)):
        if dataset is not None and "dataset" in chunk.columns:
            chunk = chunk[chunk["dataset"].astype(str) == str(dataset)]
        if panel is not None and "panel" in chunk.columns:
            chunk = chunk[chunk["panel"].astype(str) == str(panel)]
        if chunk.empty:
            continue

        present_group_cols = [c for c in group_cols if c in chunk.columns]
        if not present_group_cols:
            key = ("__all__",)
            remaining = sample_per_group - counts.get(key, 0)
            if remaining <= 0:
                continue
            take = min(len(chunk), remaining)
            kept.append(chunk.sample(n=take, random_state=rng_seed + chunk_idx))
            counts[key] = counts.get(key, 0) + take
            continue

        sampled_groups = []
        for group_key, group in chunk.groupby(present_group_cols, observed=True):
            if not isinstance(group_key, tuple):
                group_key = (group_key,)
            key = tuple(str(x) for x in group_key)
            remaining = sample_per_group - counts.get(key, 0)
            if remaining <= 0:
                continue
            take = min(len(group), remaining)
            sampled_groups.append(group.sample(n=take, random_state=rng_seed + chunk_idx))
            counts[key] = counts.get(key, 0) + take
        if sampled_groups:
            kept.append(pd.concat(sampled_groups, ignore_index=True))

    if not kept:
        return pd.DataFrame()
    return pd.concat(kept, ignore_index=True)


def count_matrix_rows_filtered(path, dataset=None, panel=None, chunksize=250_000):
    """Count matching CSV rows without materializing marker columns."""
    columns = list(pd.read_csv(path, nrows=0).columns)
    filter_cols = [c for c in ("dataset", "panel") if c in columns]
    usecols = filter_cols or [columns[0]]
    total = 0
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
        keep = np.ones(len(chunk), dtype=bool)
        if dataset is not None and "dataset" in chunk.columns:
            keep &= chunk["dataset"].astype(str).eq(str(dataset)).to_numpy()
        if panel is not None and "panel" in chunk.columns:
            keep &= chunk["panel"].astype(str).eq(str(panel)).to_numpy()
        total += int(keep.sum())
    return total


def resolve_matrix_sampling(
    path, dataset, panel, sample_per_group_cap=None
):
    total = count_matrix_rows_filtered(
        path, dataset=dataset, panel=panel, chunksize=matrix_chunksize
    )
    cap = panel_sample_per_group if sample_per_group_cap is None else sample_per_group_cap
    use_sampling = total > matrix_sampling_threshold
    sample_cap = cap if use_sampling else None
    mode = f"sample (max {sample_cap:,}/sample_name+plate)" if use_sampling else "full data"
    print(
        f"[LOAD] Source rows for {dataset}/{panel}: {total:,} | "
        f"threshold={matrix_sampling_threshold:,} -> {mode}"
    )
    return sample_cap, total


def normalize_group_labels(df):
    """Normalize biological group labels to the plotting convention used in prior figures."""
    if df is None or df.empty or "group" not in df.columns:
        return df
    df = df.copy()
    group = df["group"].astype("string").str.strip()
    group_lower = group.str.lower()
    sample = df["sample_name"].astype("string") if "sample_name" in df.columns else pd.Series("", index=df.index, dtype="string")
    sample_lower = sample.str.lower()

    healthy_mask = group_lower.isin(["healthy", "hbm", "normal", "control"])
    healthy_mask = healthy_mask | (group_lower.isin(["unknown", "nan", "none", ""]) & sample_lower.str.contains("healthy|hbm", na=False))
    df.loc[healthy_mask, "group"] = "healthy"
    df.loc[group_lower.eq("adult"), "group"] = "adult"
    df.loc[group_lower.eq("pediatric"), "group"] = "pediatric"
    return df

with open(study_config_path, encoding="utf-8-sig") as f:
    study_config = json.load(f)

config = get_config(config_dict=study_config["cytof_config"])
ctx    = CyTOFContext(config)

with open(study_config["outputs"]["markers_path"], encoding="utf-8-sig") as f:
    markers = json.load(f)

target_dataset = target_dataset_override or study_config["dataset"]
target_panel   = target_panel_override or study_config.get("panel", "Panel1")
anchor_name    = ctx.anchors.get(target_dataset)

raw_path   = study_config["outputs"]["matrix_path"]
final_path = study_config["outputs"]["matrix_corrected_path"]

effective_sample_per_group, source_cell_count = resolve_matrix_sampling(
    raw_path, target_dataset, target_panel
)

print(f"[LOAD] Reading only dataset={target_dataset}, panel={target_panel}")
df_raw = read_matrix_filtered_grouped(
    raw_path,
    dataset=target_dataset,
    panel=target_panel,
    chunksize=matrix_chunksize,
    sample_per_group=effective_sample_per_group,
    group_cols=panel_sample_group_cols,
)
df_a = read_matrix_filtered_grouped(
    final_path,
    dataset=target_dataset,
    panel=target_panel,
    chunksize=matrix_chunksize,
    sample_per_group=effective_sample_per_group,
    group_cols=panel_sample_group_cols,
)

if df_raw.empty or df_a.empty:
    empty_names = [name for name, frame in (("raw", df_raw), ("corrected", df_a)) if frame.empty]
    raise ValueError(
        f"No rows loaded from {', '.join(empty_names)} matrix for "
        f"dataset={target_dataset!r}, panel={target_panel!r}. "
        "Check that the study config and optional dataset/panel overrides match the CSV contents."
    )

# Apply arcsinh to the original matrix before comparison.
numeric_markers = [m for m in markers if m in df_raw.columns]
df_b = arcsinh(df_raw, numeric_markers)
df_b = normalize_group_labels(df_b)
df_a = normalize_group_labels(df_a)
del df_raw

print(f"Dataset  : {target_dataset}")
print(f"Panel    : {target_panel}")
print(f"Anchor   : {anchor_name}")
print(f"Before   : {len(df_b)} cells | After: {len(df_a)} cells")
print(f"Markers ({len(markers)}): {markers[:5]}...")


In [ ]:
# --- GENERATE & LOAD STUDY-SPECIFIC COLORS ---
_results_dir     = os.path.dirname(study_config["outputs"]["matrix_corrected_path"])
_study_name      = study_config.get("study_name", "study")
colors_save_path = os.path.join(_results_dir, f"colors_dict_{_study_name}.pkl")

if df_a is not None and not df_a.empty and not os.path.exists(colors_save_path):
    print("\n===== GENERATING GLOBAL COLORS FROM GATED DATA =====")
    all_cell_types = set()
    tc_dir = os.path.join(ctx.output_dir, "time_corrected")
    if os.path.exists(tc_dir):
        for f in glob(os.path.join(tc_dir, "*.csv")):
            try:
                df_temp = pd.read_csv(f, usecols=["cell_type"])
                all_cell_types.update(df_temp["cell_type"].dropna().unique())
            except: pass
    cols = ["celltypes", "panel", "plate", "sample_name", "dataset"]
    present_cols = [col for col in cols if col in df_a.columns]
    temp_df = df_a[present_cols].drop_duplicates().copy()
    for col in cols:
        if col not in temp_df.columns:
            temp_df[col] = np.nan
    temp_df["cell_type"] = pd.Series(list(all_cell_types))
    generate_and_save_colors(df=temp_df, save_path=colors_save_path)
elif os.path.exists(colors_save_path):
    print(f"[INFO] Using existing colors from {colors_save_path}")

global_colors = {}
if os.path.exists(colors_save_path):
    with open(colors_save_path, "rb") as f_in:
        global_colors = pickle.load(f_in)
    print(f"[INFO] Global colors loaded ({len(global_colors)} keys)")

In [ ]:
import gc

df_b = normalize_group_labels(df_b)
df_a = normalize_group_labels(df_a)

factors = [c for c in ["plate", "sample_name", "celltypes", "Time"] if c in df_b.columns]

# df_b and df_a are already filtered to target_panel — no loop needed
if df_b.empty or df_a.empty:
    print(f"[SKIP] No data found for panel {target_panel}")
else:
    # Restrict to true biological markers present in BOTH before and after DataFrames.
    panel_markers = [
        m for m in markers
        if m in df_b.columns
        and m in df_a.columns
        and m in ctx.true_markers
        and df_b[m].notna().any()
    ]
    print(f"Panel    : {target_panel}  |  Dataset: {target_dataset}")
    print(f"Markers  : {len(panel_markers)} | Before : {len(df_b)} | After : {len(df_a)}")

    # Per-panel colors
    current_colors = {}
    if global_colors and os.path.exists(colors_save_path):
        current_colors = load_and_subset_colors(df_a, colors_save_path)

    # Output directories
    base_panel_dir  = os.path.join(ctx.output_dir, target_dataset, target_panel)
    path_Variance   = os.path.join(base_panel_dir, "Batch_Variance")
    path_EMD        = os.path.join(base_panel_dir, "Batch_EMD")
    path_MarkerCorr = os.path.join(base_panel_dir, "Batch_MarkerCorrelation")
    path_AnchorSignal = os.path.join(base_panel_dir, "Batch_AnchorSignal")
    path_AnchorCorrection = os.path.join(base_panel_dir, "Batch_AnchorCorrection")
    for _d in [path_Variance, path_EMD, path_MarkerCorr, path_AnchorSignal, path_AnchorCorrection]:
        os.makedirs(_d, exist_ok=True)

    # ── 1. PCA Variance Analysis ──────────────────────────────────────────
    print(f"\n[ANALYZE] Factor PCA Variance...")
    var_b = variation_pca(df_b, markers=panel_markers, group_col="group",
                          info_cols=factors, panel=target_panel)
    var_a = variation_pca(df_a, markers=panel_markers, group_col="group",
                          info_cols=factors, panel=target_panel)
    print("Before correction:"); print(var_b)
    print("After  correction:"); print(var_a)

    # ── 2. Anchor Signal Alignment (across plates, before correction) ─────
    print(f"\n[ANALYZE] Anchor Signal Alignment across plates...")
    visualize_anchor_across_plates(df=df_b, panel=target_panel,
                                   markers=panel_markers, save_dir=path_AnchorSignal,
                                   anchor_name=anchor_name)
    plt.show()

    # ── 3. Time alignment across plates ───────────────────────────
    print(f"\n[ANALYZE] Time alignment across plates...")
    time_alignment_dir = os.path.join(base_panel_dir, "Batch_TimeAlignment")
    plot_time_effect_density_merged(
        before_df=df_b,
        after_df=df_a,
        panel=target_panel,
        markers=panel_markers,
        save_dir=time_alignment_dir,
    )
    plt.show()

    # ── 4. Anchor Before / After comparison ───────────────────────────────
    df_anc_b = df_b[df_b["sample_name"] == anchor_name].copy()
    df_anc_a = df_a[df_a["sample_name"] == anchor_name].copy()
    if not df_anc_b.empty and not df_anc_a.empty:
        print(f"\n[ANALYZE] Anchor before/after distribution comparison...")
        plot_signal_distributions(df_before=df_anc_b, df_after=df_anc_a,
                                  panel=target_panel, markers=panel_markers,
                                  save_dir=path_AnchorCorrection)
        plt.show()
    else:
        print(f"  [SKIP] Anchor '{anchor_name}' not found for {target_panel}")
    del df_anc_b, df_anc_a

    # ── 4. UMAPs ──────────────────────────────────────────────────────────
    print(f"\n[ANALYZE] UMAP — Plate Effect...")
    umap_plate_effect(df_b, df_a, markers=panel_markers,
                      anchor_name=anchor_name,
                      subsample=50000, save_path=None)
    plt.show()

    print(f"\n[ANALYZE] UMAP — Batch Evaluation...")
    umap_batch_evaluation(df_raw=df_b, df_corrected=df_a,
                          markers=panel_markers, save_path=None)
    plt.show()

    print(f"\n[ANALYZE] UMAP — Celltype by Plate...")
    umap_celltype_by_plate(df_raw=df_b, df_corrected=df_a,
                           markers=panel_markers, subsample=50000, save_path=None)
    plt.show()

    # ── 5. EMD / MAD ──────────────────────────────────────────────────────
    print(f"\n[ANALYZE] Computing EMD/MAD Scores...")
    cluster_col = "celltypes" if "celltypes" in df_b.columns else None
    emd_df, _, mad_score = compute_emd_mad(
        df_raw=df_b, df_corr=df_a,
        markers=panel_markers,
        ref_plate=ctx.ref_plate,
        cluster_col=cluster_col,
        anchor=anchor_name,
        n_sample=50000,
        save_dir=path_EMD,
    )
    if emd_df is not None:
        avg_mad = mad_score["MAD_score"].median() if isinstance(mad_score, pd.DataFrame) else mad_score
        print(f"  MAD Score (median): {avg_mad:.4f}")
        plot_emd(emd_df, mad_score, save_path=path_EMD)
        plot_emd_per_marker(emd_df, save_path=path_EMD)
        plt.show()

    # ── 6. Marker-level Variance Boxplot ──────────────────────────────────
    print(f"\n[ANALYZE] Marker-level variance boxplots...")
    mvar_b = variation_marker(df_b, markers=panel_markers, group_col=None,
                              info_cols=factors, panel=target_panel)
    mvar_a = variation_marker(df_a, markers=panel_markers, group_col=None,
                              info_cols=factors, panel=target_panel)
    mvar_b.to_csv(os.path.join(path_Variance,
        f"variance_marker_raw_before_{target_dataset}_{target_panel}_Batch.csv"), index=False)
    mvar_a.to_csv(os.path.join(path_Variance,
        f"variance_marker_raw_after_{target_dataset}_{target_panel}_Batch.csv"), index=False)
    plot_variance_marker_boxplot(
        var_marker_raw=mvar_b, var_marker_corr=mvar_a,
        save_path=os.path.join(path_Variance,
            f"variance_marker_boxplot_{target_dataset}_{target_panel}_Batch.png"),
        use_log=True
    )
    plt.show()

    # ── 7. Batch Correction Metrics ────────────────────────────────────────
    print(f"\n[ANALYZE] Batch Correction Metrics...")
    try:
        metrics_before, metrics_after = compute_metrics(
            df_after=df_a, df_before=df_b,
            markers=panel_markers, batch_key="plate",
            label_key="celltypes", n_iterations=20,
        )
        plot_metrics_comparison(metrics_before, metrics_after,
                                panel=target_panel, save_dir=path_Variance)
        plt.show()
    except Exception as _e:
        print(f"  [WARN] Metrics skipped: {_e}")

    # ── 8. Marker-Marker Correlation ───────────────────────────────────────
    print(f"\n[ANALYZE] Marker-Marker Correlation...")
    marker_corr(df_before=df_b, df_after=df_a,
                markers=panel_markers, save_dir=path_MarkerCorr,
                group_col="condition")
    plt.show()

    gc.collect()
    print(f"\n[DONE] Finished panel: {target_panel}")

In [ ]:
import gc

# Current panel data can be reused by the cross-panel cell below.
# To free it manually, run: del df_b, df_a; gc.collect()
gc.collect()
print("[MEMORY] Keeping current panel in memory for possible cross-panel reuse.")


In [ ]:
import gc

# Cross-Panel Variance Analysis
# D0 usually uses one study config containing both panels.
# D24 uses split configs, one file per panel. Keep those modes explicit.

# USER INPUT
p1_name_override = "Panel1"   # e.g. "Panel1"
p2_name_override = "Panel2"       # set to "Panel2" only when it really exists
cross_panel_config_mode = "auto"  # "auto", "single", or "split"
cross_panel_sample_per_group = globals().get("cross_panel_sample_per_group", panel_sample_per_group or 20000)
cross_panel_chunksize = globals().get("cross_panel_chunksize", matrix_chunksize)
cross_panel_random_state = globals().get("cross_panel_random_state", 42)
# -----------------------------------------------------------------------------

gc.collect()

p1_name = p1_name_override or "Panel1"
p2_name = p2_name_override

# Discover sibling split configs before falling back to the panels declared
# inside the current config. A Panel1 split config declares only Panel1 even
# when a matching Panel2 config exists beside it.
if p2_name is None:
    _candidate_panels = ["Panel1", "Panel2"]
    _sibling_paths = infer_panel_config_paths(
        os.path.abspath(study_config_path),
        _candidate_panels,
        current_panel=str(target_panel),
    )
    _has_split_siblings = all(
        _sibling_paths.get(p) and os.path.exists(_sibling_paths[p])
        for p in _candidate_panels
    ) and len({os.path.abspath(_sibling_paths[p]) for p in _candidate_panels}) > 1

    if _has_split_siblings:
        p1_name, p2_name = _candidate_panels
    else:
        _declared_panels = study_config.get("panels")
        if isinstance(_declared_panels, dict):
            _declared_panels = list(_declared_panels)
        elif not isinstance(_declared_panels, (list, tuple)):
            _declared_panels = [study_config.get("panel", target_panel)]
        _declared_panels = [str(p) for p in _declared_panels if p]
        p2_name = next((p for p in _declared_panels if p != str(p1_name)), None)

panel_names = [p for p in [p1_name, p2_name] if p is not None]

if cross_panel_sample_per_group is None or cross_panel_sample_per_group <= 0:
    raise ValueError("Set cross_panel_sample_per_group to a positive integer for D24 cross-panel visualization.")

vis_save_dir = os.path.join(ctx.output_dir, target_dataset, "CrossPanel_Visualization")
os.makedirs(vis_save_dir, exist_ok=True)
print(f"[INFO] Cross-panel figures -> {vis_save_dir}")

_current_cfg_path = os.path.abspath(study_config_path)
_current_panel = str(study_config.get("panel", target_panel))

if cross_panel_config_mode == "auto":
    _candidate_paths = infer_panel_config_paths(_current_cfg_path, panel_names, current_panel=_current_panel)
    _has_split_configs = all(_candidate_paths.get(p) and os.path.exists(_candidate_paths[p]) for p in panel_names)
    _same_file_for_all = len({os.path.abspath(_candidate_paths[p]) for p in panel_names if _candidate_paths.get(p)}) == 1
    cross_panel_config_mode = "split" if _has_split_configs and not _same_file_for_all else "single"

print(f"[INFO] Cross-panel config mode: {cross_panel_config_mode}")

if cross_panel_config_mode == "split":
    panel_config_paths = infer_panel_config_paths(_current_cfg_path, panel_names, current_panel=_current_panel)
elif cross_panel_config_mode == "single":
    panel_config_paths = {p: _current_cfg_path for p in panel_names}
else:
    raise ValueError('cross_panel_config_mode must be "auto", "single", or "split"')

print("[INFO] Panel config paths:")
for _p, _path in panel_config_paths.items():
    print(f"  {_p}: {_path if _path else 'NOT FOUND'}")

panel_configs = {}
for _p, _path in panel_config_paths.items():
    if not _path or not os.path.exists(_path):
        raise FileNotFoundError(f"Config for {_p} was not found. Set panel_config_paths or cross_panel_config_mode manually.")
    with open(_path, encoding="utf-8-sig") as _f:
        panel_configs[_p] = json.load(_f)

def _markers_from_config(cfg):
    with open(cfg["outputs"]["markers_path"], encoding="utf-8-sig") as _f:
        return json.load(_f)

def _valid_panel_markers(candidate_markers, *dfs):
    true_marker_set = set(getattr(ctx, "true_markers", []) or candidate_markers)
    valid = []
    for marker in candidate_markers:
        if marker not in true_marker_set:
            continue
        if all(marker in df.columns and pd.api.types.is_numeric_dtype(df[marker]) for df in dfs):
            valid.append(marker)
    return valid

def _reuse_current_panel(panel_name):
    return ("df_b" in globals() and "df_a" in globals() and str(target_panel) == str(panel_name) and not df_b.empty and not df_a.empty)

def _load_batch_pair_for_panel(panel_name):
    if _reuse_current_panel(panel_name):
        print(f"[REUSE] Using already loaded {panel_name}: before={df_b.shape}, after={df_a.shape}")
        return df_b, df_a, markers
    cfg = panel_configs[panel_name]
    panel_markers = _markers_from_config(cfg)
    print(f"[LOAD] Reading {panel_name} for cross-panel comparison")
    _panel_sample_cap, _panel_source_count = resolve_matrix_sampling(
        cfg["outputs"]["matrix_path"], target_dataset, panel_name,
        sample_per_group_cap=cross_panel_sample_per_group,
    )
    df_before = read_matrix_filtered_grouped(cfg["outputs"]["matrix_path"], dataset=target_dataset, panel=panel_name, chunksize=cross_panel_chunksize, sample_per_group=_panel_sample_cap, group_cols=panel_sample_group_cols, random_state=cross_panel_random_state)
    df_before = arcsinh(df_before, [m for m in panel_markers if m in df_before.columns])
    df_after = read_matrix_filtered_grouped(cfg["outputs"]["matrix_corrected_path"], dataset=target_dataset, panel=panel_name, chunksize=cross_panel_chunksize, sample_per_group=_panel_sample_cap, group_cols=panel_sample_group_cols, random_state=cross_panel_random_state)
    df_before = normalize_group_labels(df_before)
    df_after = normalize_group_labels(df_after)
    print(f"[LOAD] {panel_name}: before={df_before.shape}, after={df_after.shape}")
    print(f"[INFO] {panel_name} groups: before={sorted(df_before['group'].dropna().astype(str).unique()) if 'group' in df_before.columns else []}, after={sorted(df_after['group'].dropna().astype(str).unique()) if 'group' in df_after.columns else []}")
    gc.collect()
    return df_before, df_after, panel_markers

def _canonicalize_group(df):
    """Use `condition` as the biological group when `group` is absent (OHSU)."""
    if df is None:
        return df
    df = df.copy()
    if "group" not in df.columns and "condition" in df.columns:
        df["group"] = df["condition"]
    return normalize_group_labels(df)


df_p1_b, df_p1_a, markers_p1 = _load_batch_pair_for_panel(p1_name)
df_p1_b = _canonicalize_group(df_p1_b)
df_p1_a = _canonicalize_group(df_p1_a)
gc.collect()

if p2_name is not None:
    df_p2_b, df_p2_a, markers_p2 = _load_batch_pair_for_panel(p2_name)
    df_p2_b = _canonicalize_group(df_p2_b)
    df_p2_a = _canonicalize_group(df_p2_a)
else:
    df_p2_b, df_p2_a, markers_p2 = pd.DataFrame(), pd.DataFrame(), []
gc.collect()

available_panels = [p for p, df in [(p1_name, df_p1_b), (p2_name, df_p2_b)] if p is not None and not df.empty]
print(f"[INFO] Available panels: {available_panels}")
if len(available_panels) == 1:
    # Preserve the original independent recalculation mode. This is grouped
    # PCA/ANOVA variance, not the overall result from the preceding cell.
    panel_name = available_panels[0]
    single_factors = [
        c for c in ["plate", "sample_name", "celltypes", "Time"]
        if c in df_p1_b.columns and c in df_p1_a.columns
    ]
    markers_p1 = _valid_panel_markers(markers_p1, df_p1_b, df_p1_a)
    groups_before = sorted(df_p1_b["group"].dropna().astype(str).unique()) if "group" in df_p1_b else []
    groups_after = sorted(df_p1_a["group"].dropna().astype(str).unique()) if "group" in df_p1_a else []
    print(f"[INFO] {panel_name} groups: before={groups_before}, after={groups_after}")
    print(f"[ANALYZE] Recalculating grouped batch variance for {panel_name}")

    var_p1_b = variation_pca(
        df_p1_b, markers=markers_p1, group_col="group",
        info_cols=single_factors, panel=panel_name,
    )
    var_p1_a = variation_pca(
        df_p1_a, markers=markers_p1, group_col="group",
        info_cols=single_factors, panel=panel_name,
    )
    c_var_p1_b, c_var_p1_a = compute_celltype_variance(
        df_p1_b, df_p1_a, batch_col="plate", ctx=ctx,
    )

    single_vis_dir = os.path.join(ctx.output_dir, target_dataset, panel_name, "Batch_Variance")
    os.makedirs(single_vis_dir, exist_ok=True)
    global_path = os.path.join(single_vis_dir, f"grouped_variance_{target_dataset}_{panel_name}_Batch.png")
    celltype_path = os.path.join(single_vis_dir, f"celltype_variance_{target_dataset}_{panel_name}_Batch.png")
    plot_variance_bar(var_p1_b, var_p1_a, save_path=global_path)
    plot_variance_bar_celltype(c_var_p1_b, c_var_p1_a, save_path=celltype_path)

    from IPython.display import Image, display
    display(Image(filename=global_path))
    display(Image(filename=celltype_path))

    # Run the plotting helpers in their supported single-panel form and keep
    # these outputs under the panel-specific batch visualization directory.
    single_panel_dir = os.path.join(ctx.output_dir, target_dataset, panel_name, "Batch_Visualization")
    os.makedirs(single_panel_dir, exist_ok=True)

    _pheno = ctx.pheno_markers or study_config.get("cytof_config", {}).get("pheno_markers")
    umap_markers_p1 = [m for m in _pheno if m in df_p1_b.columns] if _pheno else markers_p1
    print(f"\n[POST-PROCESS] Single-panel correction UMAPs for {panel_name}...")
    visualize_shared_marker_correction(
        df_raw_P1=df_p1_b,
        df_corrected_P1=df_p1_a,
        df_raw_P2=None,
        df_corrected_P2=None,
        markers_P1=umap_markers_p1,
        markers_P2=None,
        anchor_name=anchor_name,
        save_dir=single_panel_dir,
    )
    plt.show()

    print(f"\n[POST-PROCESS] Marker-expression correlation for {panel_name}...")
    marker_expression_correlation_plot(
        df_P1_before=df_p1_b,
        df_P1_after=df_p1_a,
        df_P2_before=None,
        df_P2_after=None,
        save_dir=single_panel_dir,
        ctx=ctx,
    )
    plt.show()
    print(f"[DONE] Single-panel batch figures saved to: {single_panel_dir}")
else:
    factors = [c for c in ["plate", "sample_name", "celltypes", "Time"] if c in df_p1_b.columns or c in df_p2_b.columns]

    # Resolve pheno_markers for UMAPs (same logic as pipeline)
    _pheno = ctx.pheno_markers or study_config.get("cytof_config", {}).get("pheno_markers")

    try:
        # --- Panel 1 ---
        print(f"\n[ANALYZE] PCA for {p1_name}...")
        # Keep only the markers in this panel
        df_p1_b = df_p1_b.dropna(axis=1, how="all")
        df_p1_a = df_p1_a.dropna(axis=1, how="all")

        if df_p1_b.empty or df_p1_a.empty:
            raise ValueError(f"No data for {p1_name} in one or both matrices.")

        markers_p1 = _valid_panel_markers(markers_p1, df_p1_b, df_p1_a)
        var_p1_b = variation_pca(df_p1_b, markers=markers_p1, group_col="group", info_cols=factors, panel=p1_name)
        var_p1_a = variation_pca(df_p1_a, markers=markers_p1, group_col="group", info_cols=factors, panel=p1_name)
        c_var_p1_b, c_var_p1_a = compute_celltype_variance(df_p1_b, df_p1_a, batch_col="plate", ctx=ctx)

        # --- Panel 2 ---
        print(f"\n[ANALYZE] PCA for {p2_name}...")
        # Keep only the markers in this panel
        df_p2_b = df_p2_b.dropna(axis=1, how="all")
        df_p2_a = df_p2_a.dropna(axis=1, how="all")

        if df_p2_b.empty or df_p2_a.empty:
            raise ValueError(f"No data for {p2_name} in one or both matrices.")

        markers_p2 = _valid_panel_markers(markers_p2, df_p2_b, df_p2_a)
        var_p2_b = variation_pca(df_p2_b, markers=markers_p2, group_col="group", info_cols=factors, panel=p2_name)
        var_p2_a = variation_pca(df_p2_a, markers=markers_p2, group_col="group", info_cols=factors, panel=p2_name)
        c_var_p2_b, c_var_p2_a = compute_celltype_variance(df_p2_b, df_p2_a, batch_col="plate", ctx=ctx)

        # Pheno-filtered marker lists for UMAP (intersect with shared markers)
        if _pheno:
            umap_m1 = [m for m in _pheno if m in df_p1_b.columns]
            umap_m2 = [m for m in _pheno if m in df_p2_b.columns]
        else:
            umap_m1 = markers_p1
            umap_m2 = markers_p2

        # --- Cross-Panel Plots (saved to vis_save_dir) ---
        print(f"\n[POST-PROCESS] Cross-panel variance: {p1_name} vs {p2_name}")
        plot_variance_bar(var_p1_b, var_p1_a, var_p2_b, var_p2_a,
                          save_path=os.path.join(vis_save_dir, "global_variance_bar.png"))
        plt.show()

        plot_variance_bar_celltype(c_var_p1_b, c_var_p1_a, c_var_p2_b, c_var_p2_a,
                                   save_path=os.path.join(vis_save_dir, "celltype_variance_bar.png"))
        plt.show()

        print(f"\n[POST-PROCESS] Shared marker correction visualization...")
        visualize_shared_marker_correction(
            df_p1_b, df_p1_a, df_p2_b, df_p2_a,
            umap_m1, umap_m2,
            anchor_name=anchor_name,
            save_dir=vis_save_dir,
        )

        marker_expression_correlation_plot(
            df_p1_b, df_p1_a, df_p2_b, df_p2_a,
            save_dir=vis_save_dir
        )

        del df_p1_b, df_p1_a, df_p2_b, df_p2_a
        gc.collect()
        print(f"\n[DONE] Cross-panel figures saved to: {vis_save_dir}")

    except Exception as e:
        import traceback
        print(f"[ERROR] Cross-panel visualization failed: {e}")
        traceback.print_exc()

gc.collect()


## Step 8 — Cross-Dataset Comparison

Compare batch-corrected outputs from **two different study configs** (e.g. D0 vs D24, or cohort_1 vs cohort_2).  
Only markers shared between both datasets are used. Set `panel_filter` to restrict to a specific panel, or leave `None` to use all panels.

In [ ]:
import gc

# Step 8: Cross-Dataset Comparison
# Loads corrected matrices from two study configs and compares shared markers.

# ── USER INPUT ────────────────────────────────────────────────────────────────
study_config_path_ds1 = "../results/cohort_1/study_config_cohort1_D0.json"
study_config_path_ds2 = "../results/cohort_1/study_config_cohort1_D24.json"

# Optional: restrict to a single panel (e.g. "Panel1"). None = use all panels.
panel_filter = None

# Number of cells to subsample per dataset for heavy plots (None = no limit)
cross_ds_subsample = 50000
# ─────────────────────────────────────────────────────────────────────────────

gc.collect()

with open(study_config_path_ds1, encoding="utf-8-sig") as _f:
    _sc1 = json.load(_f)
with open(study_config_path_ds2, encoding="utf-8-sig") as _f:
    _sc2 = json.load(_f)

_ds1_label = _sc1.get("study_name") or _sc1.get("dataset", "Dataset1")
_ds2_label = _sc2.get("study_name") or _sc2.get("dataset", "Dataset2")

print(f"[INFO] Loading corrected matrices...")
print(f"  DS1: {_ds1_label}  →  {_sc1['outputs']['matrix_corrected_path']}")
print(f"  DS2: {_ds2_label}  →  {_sc2['outputs']['matrix_corrected_path']}")

df_ds1 = pd.read_csv(_sc1["outputs"]["matrix_corrected_path"])
df_ds2 = pd.read_csv(_sc2["outputs"]["matrix_corrected_path"])

# Load per-study markers (biological markers only)
with open(_sc1["outputs"]["markers_path"], encoding="utf-8-sig") as _f:
    _markers1 = json.load(_f)
with open(_sc2["outputs"]["markers_path"], encoding="utf-8-sig") as _f:
    _markers2 = json.load(_f)

# Optionally filter to a single panel
if panel_filter:
    if "panel" in df_ds1.columns:
        df_ds1 = df_ds1[df_ds1["panel"] == panel_filter].copy()
    if "panel" in df_ds2.columns:
        df_ds2 = df_ds2[df_ds2["panel"] == panel_filter].copy()
    print(f"[INFO] Panel filter applied: {panel_filter}")

# Shared markers present in both datasets
_shared_markers = sorted(
    set(_markers1) & set(_markers2) & set(df_ds1.columns) & set(df_ds2.columns)
)
print(f"[INFO] Shared markers ({len(_shared_markers)}): {_shared_markers[:8]}...")

if not _shared_markers:
    print("[SKIP] No shared markers found between the two datasets.")
else:
    # Subsample for performance
    if cross_ds_subsample and len(df_ds1) > cross_ds_subsample:
        df_ds1 = df_ds1.sample(cross_ds_subsample, random_state=42)
    if cross_ds_subsample and len(df_ds2) > cross_ds_subsample:
        df_ds2 = df_ds2.sample(cross_ds_subsample, random_state=42)

    print(f"[INFO] Cells: {_ds1_label}={len(df_ds1)}, {_ds2_label}={len(df_ds2)}")

    # ── 1. Shared-marker distribution comparison (violin / ridge) ──────────────
    n_cols = min(4, len(_shared_markers))
    n_rows = int(np.ceil(len(_shared_markers) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
    axes = np.array(axes).flatten()

    for i, marker in enumerate(_shared_markers):
        ax = axes[i]
        vals1 = df_ds1[marker].dropna().values
        vals2 = df_ds2[marker].dropna().values
        ax.hist(vals1, bins=60, alpha=0.55, density=True, color=_PANEL1_COLOR, label=_ds1_label)
        ax.hist(vals2, bins=60, alpha=0.55, density=True, color=_PANEL2_COLOR, label=_ds2_label)
        ax.set_title(marker, fontsize=9)
        ax.set_xlabel("arcsinh", fontsize=7)
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Cross-Dataset: {_ds1_label} vs {_ds2_label} — Shared Markers", fontsize=12)
    fig.tight_layout()
    _cross_ds_dir = os.path.join(os.path.dirname(_sc1["outputs"]["matrix_corrected_path"]), "cross_dataset_visualization")
    os.makedirs(_cross_ds_dir, exist_ok=True)
    fig.savefig(os.path.join(_cross_ds_dir, f"cross_dataset_shared_markers_{_ds1_label}_vs_{_ds2_label}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    # ── 2. Median expression heatmap per dataset ───────────────────────────────
    _med1 = df_ds1[_shared_markers].median().rename(_ds1_label)
    _med2 = df_ds2[_shared_markers].median().rename(_ds2_label)
    _med_df = pd.DataFrame([_med1, _med2])

    fig, ax = plt.subplots(figsize=(max(8, len(_shared_markers) * 0.5), 3))
    sns.heatmap(_med_df, ax=ax, cmap="viridis", annot=True, fmt=".2f",
                annot_kws={"size": 7}, linewidths=0.3)
    ax.set_title(f"Median Expression: {_ds1_label} vs {_ds2_label}", fontsize=11)
    ax.set_xlabel("Marker")
    fig.tight_layout()
    fig.savefig(os.path.join(_cross_ds_dir, f"cross_dataset_median_heatmap_{_ds1_label}_vs_{_ds2_label}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    # ── 3. PCA variance by dataset factor ─────────────────────────────────────
    df_ds1["_dataset_label"] = _ds1_label
    df_ds2["_dataset_label"] = _ds2_label
    df_combined = pd.concat([df_ds1, df_ds2], ignore_index=True)

    _factors_avail = [c for c in ["plate", "group", "celltypes", "_dataset_label"] if c in df_combined.columns]
    _combined_panel = panel_filter or "AllPanels"
    var_combined = variation_pca(df_combined, markers=_shared_markers,
                                 group_col="_dataset_label",
                                 info_cols=_factors_avail,
                                 panel=_combined_panel)
    print(f"\nPCA Variance by factor (combined):")
    print(var_combined)

    del df_ds1, df_ds2, df_combined
    gc.collect()
    print(f"\n[DONE] Cross-dataset comparison saved to: {_cross_ds_dir}")